In [ ]:
import casadi as cs
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Quadrotor Dynamics
class Quadrotor:
    def __init__(self, quad_params):
        self.quad = quad_params
        self.x = cs.MX.sym('x', 13)  # State: [p, q, v, r]
        self.u = cs.MX.sym('u', 4)   # Control inputs

    def quad_dynamics(self):
        x_dot = cs.vertcat(self.p_dynamics(), self.q_dynamics(), self.v_dynamics(), self.w_dynamics())
        return cs.Function('x_dot', [self.x, self.u], [x_dot], ['x', 'u'], ['x_dot'])

    def p_dynamics(self):
        return self.x[6:9]  # Position derivative (velocity)

    def q_dynamics(self):
        return 0.5 * cs.mtimes(skew_symmetric(self.x[9:12]), self.x[3:7])

    def v_dynamics(self):
        f_thrust = self.u * self.quad['max_thrust']
        g = cs.vertcat(0.0, 0.0, 9.81)
        a_thrust = cs.vertcat(0.0, 0.0, cs.sum1(f_thrust)) / self.quad['mass']
        return v_dot_q(a_thrust, self.x[3:7]) - g

    def w_dynamics(self):
        f_thrust = self.u * self.quad['max_thrust']
        y_f = cs.MX(self.quad['y_f'])
        x_f = cs.MX(self.quad['x_f'])
        c_f = cs.MX(self.quad['z_l_tau'])
        return cs.vertcat(
            (cs.mtimes(f_thrust.T, y_f) + (self.quad['J'][1] - self.quad['J'][2]) * self.x[11] * self.x[12]) / self.quad['J'][0],
            (-cs.mtimes(f_thrust.T, x_f) + (self.quad['J'][2] - self.quad['J'][0]) * self.x[12] * self.x[10]) / self.quad['J'][1],
            (cs.mtimes(f_thrust.T, c_f) + (self.quad['J'][0] - self.quad['J'][1]) * self.x[10] * self.x[11]) / self.quad['J'][2]
        )

# Provided helper functions remain unchanged
def v_dot_q(v, q):
    rot_mat = q_to_rot_mat(q)
    if isinstance(q, np.ndarray):
        return rot_mat.dot(v)
    return cs.mtimes(rot_mat, v)

def q_to_rot_mat(q):
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    if isinstance(q, np.ndarray):
        return np.array([
            [1 - 2 * (qy**2 + qz**2), 2 * (qx * qy - qw * qz), 2 * (qx * qz + qw * qy)],
            [2 * (qx * qy + qw * qz), 1 - 2 * (qx**2 + qz**2), 2 * (qy * qz - qw * qx)],
            [2 * (qx * qz - qw * qy), 2 * (qy * qz + qw * qx), 1 - 2 * (qx**2 + qy**2)]
        ])
    return cs.vertcat(
        cs.horzcat(1 - 2 * (qy**2 + qz**2), 2 * (qx * qy - qw * qz), 2 * (qx * qz + qw * qy)),
        cs.horzcat(2 * (qx * qy + qw * qz), 1 - 2 * (qx**2 + qz**2), 2 * (qy * qz - qw * qx)),
        cs.horzcat(2 * (qx * qz - qw * qy), 2 * (qy * qz + qw * qx), 1 - 2 * (qx**2 + qy**2))
    )

def skew_symmetric(v):
    if isinstance(v, np.ndarray):
        return np.array([[0, -v[0], -v[1], -v[2]],
                         [v[0], 0, v[2], -v[1]],
                         [v[1], -v[2], 0, v[0]],
                         [v[2], v[1], -v[0], 0]])
    return cs.vertcat(
        cs.horzcat(0, -v[0], -v[1], -v[2]),
        cs.horzcat(v[0], 0, v[2], -v[1]),
        cs.horzcat(v[1], -v[2], 0, v[0]),
        cs.horzcat(v[2], v[1], -v[0], 0))

# Quadrotor parameters and object creation
quad_params = {
    'mass': 1.0,
    'max_thrust': 5.0,
    'y_f': np.array([0.1, -0.1, -0.1, 0.1]),
    'x_f': np.array([0.1, 0.1, -0.1, -0.1]),
    'z_l_tau': np.array([-0.05, 0.05, -0.05, 0.05]),
    'J': np.array([0.01, 0.01, 0.02])
}
quad = Quadrotor(quad_params)
quad_dyn = quad.quad_dynamics()

# iLQR Implementation
def f(x, u, params, dt):
    """Discrete-time dynamics using Euler integration."""
    x_dot = np.array(quad_dyn(x=cs.DM(x), u=cs.DM(u))['x_dot']).flatten()
    return x + dt * x_dot

def compute_total_cost(x_traj, u_seq, Q, R):
    """Compute total cost along trajectory."""
    cost = 0
    horizon = u_seq.shape[0]
    for t in range(horizon):
        cost += x_traj[t].T @ Q @ x_traj[t] + u_seq[t].T @ R @ u_seq[t]
    cost += x_traj[-1].T @ Q @ x_traj[-1]
    return cost

def linearize_dynamics(x, u, params, dt, eps=1e-5):
    """Finite-difference linearization of f(x,u) around (x,u)."""
    n = len(x)
    m = len(u)
    A = np.zeros((n, n))
    B = np.zeros((n, m))
    # Compute A
    for i in range(n):
        dx = np.zeros(n)
        dx[i] = eps
        f_plus = f(x + dx, u, params, dt)
        f_minus = f(x - dx, u, params, dt)
        A[:, i] = (f_plus - f_minus) / (2 * eps)
    # Compute B
    for j in range(m):
        du = np.zeros(m)
        du[j] = eps
        f_plus = f(x, u + du, params, dt)
        f_minus = f(x, u - du, params, dt)
        B[:, j] = (f_plus - f_minus) / (2 * eps)
    return A, B

def ilqr(x0, u_init, Q, R, horizon, params, dt, u_bounds, max_iter=100, tol=1e-6):
    """
    iLQR algorithm.
    x0: initial state (n,)
    u_init: initial control sequence, shape (horizon, m)
    Q: state cost matrix (n×n)
    R: control cost matrix (m×m)
    horizon: prediction horizon (integer)
    u_bounds: tuple (min, max) for control inputs (applied elementwise)
    """
    n = len(x0)
    m = u_init.shape[1]
    # Forward rollout with initial control sequence
    x_traj = [x0.copy()]
    for t in range(horizon):
        x_next = f(x_traj[t], u_init[t], params, dt)
        x_traj.append(x_next)
    x_traj = np.array(x_traj)
    cost_prev = compute_total_cost(x_traj, u_init, Q, R)
    
    for iteration in range(max_iter):
        # Linearize dynamics along the nominal trajectory
        A_list = []
        B_list = []
        for t in range(horizon):
            A, B = linearize_dynamics(x_traj[t], u_init[t], params, dt)
            A_list.append(A)
            B_list.append(B)
        
        # Backward pass: compute value function approximations
        V_x = 2 * Q @ x_traj[-1]
        V_xx = 2 * Q
        k_list = [None] * horizon  # feedforward terms (m,)
        K_list = [None] * horizon  # feedback gains (m×n)
        diverged = False
        for t in reversed(range(horizon)):
            x_t = x_traj[t]
            u_t = u_init[t]
            L_x = 2 * Q @ x_t
            L_u = 2 * R @ u_t
            L_xx = 2 * Q
            L_uu = 2 * R
            L_ux = np.zeros((m, n))
            A = A_list[t]
            B = B_list[t]
            Q_x = L_x + A.T @ V_x
            Q_u = L_u + B.T @ V_x
            Q_xx = L_xx + A.T @ V_xx @ A
            Q_ux = L_ux + B.T @ V_xx @ A
            Q_uu = L_uu + B.T @ V_xx @ B
            # Regularization for positive definiteness
            Q_uu_reg = Q_uu + 1e-6 * np.eye(m)
            try:
                inv_Q_uu = np.linalg.inv(Q_uu_reg)
            except np.linalg.LinAlgError:
                diverged = True
                break
            k = - inv_Q_uu @ Q_u  # feedforward (m,)
            K = - inv_Q_uu @ Q_ux  # feedback (m×n)
            k_list[t] = k
            K_list[t] = K
            V_x = Q_x + K.T @ Q_uu @ k + K.T @ Q_u + Q_ux.T @ k
            V_xx = Q_xx + K.T @ Q_uu @ K + K.T @ Q_ux + Q_ux.T @ K
            V_xx = 0.5 * (V_xx + V_xx.T)
        
        if diverged:
            break
        
        # Forward pass with line search
        alpha = 1.0
        found_better = False
        for ls in range(10):
            x_new = [x0.copy()]
            u_new = []
            for t in range(horizon):
                delta_x = x_new[t] - x_traj[t]
                du = alpha * k_list[t] + K_list[t] @ delta_x
                u_new_t = u_init[t] + du
                u_new_t = np.clip(u_new_t, u_bounds[0], u_bounds[1])
                u_new.append(u_new_t)
                x_next = f(x_new[t], u_new_t, params, dt)
                x_new.append(x_next)
            u_new = np.array(u_new)
            x_new = np.array(x_new)
            cost_new = compute_total_cost(x_new, u_new, Q, R)
            if cost_new < cost_prev:
                found_better = True
                break
            alpha *= 0.5
        if not found_better:
            break
        if abs(cost_prev - cost_new) < tol:
            u_init = u_new
            x_traj = x_new
            cost_prev = cost_new
            break
        u_init = u_new
        x_traj = x_new
        cost_prev = cost_new
    return u_init, x_traj

# Receding-Horizon iLQR Simulation
def simulate_ilqr(initial_state, Q, R, horizon, sim_time, params, dt, u_bounds=(-10,10), max_iter=100, tol=1e-6):
    state = np.array(initial_state)
    trajectory = [state]
    control_history = []
    steps = int(sim_time / dt)
    for _ in range(steps):
        # Initialize control sequence: horizon steps, 4 controls per step
        u_init = np.zeros((horizon, 4))
        u_opt, x_traj = ilqr(state, u_init, Q, R, horizon, params, dt, u_bounds, max_iter, tol)
        u = u_opt[0]  # Apply only the first control action
        control_history.append(u)
        # Update state using quadrotor dynamics (via CasADi)
        x_dot = np.array(quad_dyn(x=cs.DM(state), u=cs.DM(u))['x_dot']).flatten()
        state = state + dt * x_dot
        trajectory.append(state)
    return np.array(trajectory), np.array(control_history)

# Main Evaluation Loop
trajectories = np.load('fractional_system_trajectories.npy')
U_optimal = np.load('optimal_control_U.npy')
Q_matrices = np.load('LQR_Q.npy')
R_matrices = np.load('LQR_R.npy')

# Simulation parameters
dt = 0.1
time_horizon = 8  # Prediction horizon for iLQR
sim_time = 1.6
u_bounds = (-1, 1)

mses = []
maes = []

start_time = time.time()

for i in range(2900, 2910):
    x0 = trajectories[i, 0]
    Q = Q_matrices[i]
    R = R_matrices[i]
    
    trajectory_est, estimated_controls_est = simulate_ilqr(x0, Q, R, time_horizon, sim_time, quad_params, dt, u_bounds)
    
    mae = mean_absolute_error(estimated_controls_est, U_optimal[i])
    mse = mean_squared_error(estimated_controls_est, U_optimal[i])
    print("Iteration", i, "MAE:", mae, "MSE:", mse)
    maes.append(mae)
    mses.append(mse)

end_time = time.time()
total_runtime = end_time - start_time
print("Total runtime: {:.2f} seconds".format(total_runtime))

Iteration 2900 MAE: 0.7055255256077141 MSE: 0.8072644286548858
Iteration 2901 MAE: 1.0690907077522676 MSE: 1.2513830357655018
Iteration 2902 MAE: 0.7351190732802301 MSE: 0.7485714721743103
Iteration 2903 MAE: 0.6655856984403021 MSE: 1.0759901130495548
Iteration 2904 MAE: 0.9206913412028308 MSE: 0.9428138480757333
Iteration 2905 MAE: 0.2867724865823943 MSE: 0.17009657675325915
Iteration 2906 MAE: 1.1289404711749174 MSE: 1.453750257943726
Iteration 2907 MAE: 0.6015821050053962 MSE: 0.6601392639165593
Iteration 2908 MAE: 0.5342958561156022 MSE: 0.445731862438279
Iteration 2909 MAE: 0.7986671865938177 MSE: 0.8518139470193795
Total runtime: 16.83 seconds
